# Chapter 3: LLMs for Text Classification and Generation
**Module 04 – Introduction to LLMs in Python**

> *Instructor: Iván Palomares Carrascosa, PhD — Senior Data Science & AI Manager*

## 3.1 Two Ways to Load Pre-trained LLMs

| Approach | Pros | Cons |
|---|---|---|
| **`pipeline()`** | Simple, high-level, automatic model/tokenizer selection | Less control, limited flexibility |
| **`AutoModel` / `AutoTokenizer`** | Full control, supports fine-tuning | More complex setup |

## 3.2 The AutoModel and AutoTokenizer Classes

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

model_name = "bert-base-uncased"

# Load tokenizer and model by name
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "I am an example sequence for text classification."

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64
)

print("Tokenized keys:", list(inputs.keys()))
print("Input IDs:", inputs['input_ids'])

# Forward pass through BERT
with torch.no_grad():
    outputs = model(**inputs)

print("\nHidden states size:",  outputs.last_hidden_state.shape)  # (1, seq_len, 768)
print("Pooled output size:",   outputs.pooler_output.shape)       # (1, 768)

## 3.3 Adding a Classification Head

In [ ]:
class SimpleClassifier(nn.Module):
    """Classification head on top of BERT."""
    def __init__(self, input_size, num_classes):
        super(SimpleClassifier, self).__init__()
        self.fc = nn.Linear(input_size, num_classes)

    def forward(self, x):
        return self.fc(x)


# Attach classifier to BERT pooler output (size 768)
classifier_head = SimpleClassifier(
    input_size=outputs.pooler_output.size(-1),  # 768
    num_classes=2
)

# Full forward pass
bert_repr = outputs.pooler_output  # (1, 768)
logits = classifier_head(bert_repr)
print(f"Logits: {logits}")
print(f"Predicted class: {torch.argmax(logits).item()}")

## 3.4 Text Generation with AutoModelForCausalLM

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# GPT-2 for text generation
gen_model_name = "gpt2"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForCausalLM.from_pretrained(gen_model_name)

prompt = "Deep learning is transforming"
inputs = gen_tokenizer(prompt, return_tensors="pt")

# Generate text
with torch.no_grad():
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.7,
        do_sample=True,
        pad_token_id=gen_tokenizer.eos_token_id
    )

generated = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated text:\n", generated)

## Summary

| API | Use When |
|---|---|
| `pipeline()` | Quick prototyping and demos |
| `AutoModel` | Need access to hidden states for fine-tuning |
| `AutoModelForCausalLM` | Text generation tasks |
| `pooler_output` | Aggregate BERT representation for classification |